# P1 v6 — KenLM Inference

Loads saved 2-group LoRA weights, pads CTC heads from 30→32 to fix
vocab mismatch, runs Bayesian α/β search on val set, final eval on test.

**No training — inference only.**

In [2]:
import os, warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("pyctcdecode").setLevel(logging.ERROR)
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"]                = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"]      = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]           = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"]     = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]         = "/kaggle/temp/.cache"
for p in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
           os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
           os.environ["XDG_CACHE_HOME"]]:
    os.makedirs(p, exist_ok=True)
print("Env ready.")

Env ready.


In [ ]:
!pip install pyctcdecode==0.5.0
!pip install kenlm
!pip install numpy==1.26.4
!pip -q install -U transformers datasets evaluate jiwer soundfile huggingface_hub scikit-optimize safetensors

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import evaluate
import json
import random
from itertools import groupby
from collections import Counter
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset
from transformers import WavLMForCTC, Wav2Vec2Processor
from pyctcdecode import build_ctcdecoder
from huggingface_hub import hf_hub_download

print("torch:", torch.__version__)
print("GPU:",   torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
GPU: True
GPU: Tesla T4


## Step 1 — Config

In [12]:
B2_PATH      = "/kaggle/input/datasets/kavishk/b2-pretrain/kaggle/working/b2v2_wavlm_ctc_final"
MLP_PT       = "/kaggle/input/datasets/kavishk/severity-mlp/p2_severity_mlp.pt"
LORA_PT      = "/kaggle/input/datasets/kavishk/2-group-lora-weights/p1v6_two_group_lora.pt"
# Adjust LORA_PT to wherever you uploaded p1v6_two_group_lora.pt

RANDOM_SEED     = 42
TEST_SPEAKERS   = {"F01", "F04", "FC01", "M05"}
MLP_VAL_FRAC    = 0.2
CONTROL_TARGET  = 1500
MAX_AUDIO       = 240_000
NUM_SEV_CLASSES = 4
NUM_GROUPS      = 2
BEAM_WIDTH      = 100
BATCH           = 8

TORGO_SEV_TO_INT = {"control": 0, "mild": 1, "moderate": 2, "severe": 3}
INT_TO_SEV       = {v: k for k, v in TORGO_SEV_TO_INT.items()}
SEVERITY_MAPPING = {
    "F01": "severe",   "M01": "severe",   "M02": "severe",   "M04": "severe",
    "M05": "moderate", "F03": "moderate",
    "F04": "mild",     "M03": "mild",
    "FC01": "control", "FC02": "control", "FC03": "control",
    "MC01": "control", "MC02": "control", "MC03": "control", "MC04": "control",
}
GROUP_NAMES  = ["mild_ctrl", "mod_sev"]
SEV_TO_GROUP = {"control": 0, "mild": 0, "moderate": 1, "severe": 1}

print("Config ready.")

Config ready.


## Step 2 — Load TORGO, create splits

In [8]:
from datasets import load_dataset

print("Loading TORGO...")
cache = os.environ["HF_DATASETS_CACHE"]
ds    = load_dataset("abnerh/TORGO-database", cache_dir=cache)
df    = ds["train"].to_pandas()
df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"]    = df["audio_path"].apply(lambda p: str(p).split("_")[0])
df["Category"]   = df["speaker"].map(SEVERITY_MAPPING)
df["Group"]      = df["Category"].map(SEV_TO_GROUP)

hf_full = ds["train"].add_column("speaker",  df["speaker"].tolist())
hf_full = hf_full.add_column("Category", df["Category"].tolist())
hf_full = hf_full.add_column("Group",    df["Group"].tolist())

test_mask  = df["speaker"].isin(TEST_SPEAKERS)
torgo_test = hf_full.select(df[test_mask].index.tolist())
torgo_test = torgo_test.filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
)

train_pool_df = df[~test_mask].copy()
ctrl  = train_pool_df[train_pool_df["Category"] == "control"].sample(
    n=min(CONTROL_TARGET, (train_pool_df["Category"] == "control").sum()),
    random_state=RANDOM_SEED
)
other = train_pool_df[train_pool_df["Category"] != "control"]
train_pool_df = pd.concat([ctrl, other]).sample(frac=1, random_state=RANDOM_SEED)

random.seed(RANDOM_SEED)
val_idx = []
for sev in ["control", "mild", "moderate", "severe"]:
    sev_idx = train_pool_df[train_pool_df["Category"] == sev].index.tolist()
    random.shuffle(sev_idx)
    n_val = max(1, int(len(sev_idx) * MLP_VAL_FRAC))
    val_idx.extend(sev_idx[:n_val])

torgo_val = hf_full.select(val_idx).filter(
    lambda x: len(x["audio"]["array"]) <= MAX_AUDIO, num_proc=1
)

print(f"Test: {len(torgo_test)}  Val: {len(torgo_val)}")

Loading TORGO...


Filter (num_proc=1):   0%|          | 0/1115 [00:00<?, ? examples/s]

Test: 1786  Val: 1111


## Step 3 — Load B2 (frozen)

In [9]:
processor = Wav2Vec2Processor.from_pretrained(B2_PATH)
b2_model  = WavLMForCTC.from_pretrained(
    B2_PATH, ctc_loss_reduction="mean", ctc_zero_infinity=True
)
for param in b2_model.parameters():
    param.requires_grad = False
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
b2_model = b2_model.to(device)
b2_model.eval()

# Full vocab size including special tokens — must be 32
VOCAB_SIZE = len(processor.tokenizer.get_vocab())  # 32
print(f"B2 loaded on {device} — fully frozen")
print(f"Full vocab size: {VOCAB_SIZE}")

Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

B2 loaded on cuda — fully frozen
Full vocab size: 32


## Step 4 — Preprocessing helpers

In [10]:
def prepare_torgo(batch):
    audio = batch["audio"]
    arr   = audio["array"]
    if len(arr) > MAX_AUDIO:
        arr = arr[:MAX_AUDIO]
    inputs = processor(arr, sampling_rate=audio["sampling_rate"])
    batch["input_values"]   = inputs.input_values[0]
    batch["labels"]         = processor.tokenizer(
        batch["transcription"].lower().strip()
    ).input_ids
    batch["severity_label"] = TORGO_SEV_TO_INT[batch["Category"]]
    return batch


@dataclass
class DataCollatorCTC:
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_feats  = [{"input_values": f["input_values"]} for f in features]
        batch        = self.processor.feature_extractor.pad(
            input_feats, padding=self.padding, return_tensors="pt"
        )
        label_feats  = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_feats, padding=self.padding, return_tensors="pt"
        )
        batch["labels"] = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        return batch


data_collator = DataCollatorCTC(processor=processor)

print("Preprocessing test and val...")
test_p = torgo_test.map(
    prepare_torgo, remove_columns=torgo_test.column_names, num_proc=1
)
test_p = [test_p[i] for i in range(len(test_p))]
test_cats_filtered = list(torgo_test["Category"])
print(f"test_p: {len(test_p)} utterances")

torgo_val_p = torgo_val.map(
    prepare_torgo, remove_columns=torgo_val.column_names, num_proc=1
)
torgo_val_p = [torgo_val_p[i] for i in range(len(torgo_val_p))]
val_cats_filtered = list(torgo_val["Category"])
print(f"torgo_val_p: {len(torgo_val_p)} utterances")

Preprocessing test and val...


Map (num_proc=1):   0%|          | 0/1786 [00:00<?, ? examples/s]

test_p: 1786 utterances


Map (num_proc=1):   0%|          | 0/1111 [00:00<?, ? examples/s]

torgo_val_p: 1111 utterances


## Step 5 — Build model architecture + load saved weights

In [15]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.weight = linear.weight
        self.bias   = linear.bias
        self.weight.requires_grad = False
        if self.bias is not None: self.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(rank, linear.in_features))
        self.lora_B = nn.Parameter(torch.zeros(linear.out_features, rank))
        self.scale  = alpha / rank
        nn.init.normal_(self.lora_A)

    def forward(self, x):
        return (F.linear(x, self.weight, self.bias)
                + F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scale)


def inject_lora(encoder, rank, alpha):
    PROJ_NAMES = ["out_proj", "gru_rel_pos_linear"]
    targets    = []
    for _, module in encoder.named_modules():
        for proj in PROJ_NAMES:
            child = getattr(module, proj, None)
            if child is None:
                continue
            # Skip if already any kind of LoRALinear (check by name to avoid
            # class identity mismatch when LoRALinear was defined in a previous session)
            if type(child).__name__ == "LoRALinear":
                continue
            if hasattr(child, "weight") and hasattr(child, "in_features"):
                targets.append((module, proj))

    for parent, proj in targets:
        setattr(parent, proj, LoRALinear(getattr(parent, proj), rank, alpha))
    print(f"  Replaced {len(targets)} linear layers")
    return encoder


class TwoGroupLoRAModel(nn.Module):
    def __init__(self, b2_model, vocab_size, rank=16, alpha=32):
        super().__init__()
        hidden = b2_model.config.hidden_size  # 768

        self.encoder = b2_model.wavlm
        for p in self.encoder.parameters(): p.requires_grad = False

        print("Injecting LoRA...")
        self.encoder = inject_lora(self.encoder, rank, alpha)

        # Find LoRALinear layers by name — handles class identity mismatch
        # across kernel sessions where the same class may be defined twice
        self._lora_layers = [
            m for m in self.encoder.modules()
            if type(m).__name__ == "LoRALinear"
        ]
        print(f"  {len(self._lora_layers)} LoRALinear layers found")

        self.adapter_A = nn.ParameterList([
            nn.ParameterList([nn.Parameter(l.lora_A.data.clone())
                              for l in self._lora_layers])
            for _ in range(NUM_GROUPS)
        ])
        self.adapter_B = nn.ParameterList([
            nn.ParameterList([nn.Parameter(l.lora_B.data.clone())
                              for l in self._lora_layers])
            for _ in range(NUM_GROUPS)
        ])

        # CTC heads at full vocab_size=32 for KenLM compatibility
        self.ctc_heads = nn.ModuleList([
            nn.Linear(hidden, vocab_size, bias=True) for _ in range(NUM_GROUPS)
        ])

        self.pad_token_id  = b2_model.config.pad_token_id
        self._active_group = 0
        self._load_group(0)

    def _load_group(self, g):
        for i, l in enumerate(self._lora_layers):
            l.lora_A.data.copy_(self.adapter_A[g][i].data)
            l.lora_B.data.copy_(self.adapter_B[g][i].data)
        self._active_group = g

    def _save_group(self, g):
        for i, l in enumerate(self._lora_layers):
            self.adapter_A[g][i].data.copy_(l.lora_A.data)
            self.adapter_B[g][i].data.copy_(l.lora_B.data)

    def set_active_group(self, g):
        if g == self._active_group: return
        self._save_group(self._active_group)
        self._load_group(g)

    def forward_single(self, iv, am, g):
        self.set_active_group(g)
        out    = self.encoder(iv, attention_mask=am, output_hidden_states=True)
        logits = self.ctc_heads[g](out.last_hidden_state)
        return logits, out.hidden_states


print("Building model...")
b2_model.cpu()
lora_model = TwoGroupLoRAModel(b2_model, vocab_size=VOCAB_SIZE)

print(f"\nLoading saved weights from {LORA_PT}...")
ckpt = torch.load(LORA_PT, map_location="cpu")

for i, p in enumerate(lora_model.adapter_A[0]): p.data.copy_(ckpt["A0"][i])
for i, p in enumerate(lora_model.adapter_B[0]): p.data.copy_(ckpt["B0"][i])
for i, p in enumerate(lora_model.adapter_A[1]): p.data.copy_(ckpt["A1"][i])
for i, p in enumerate(lora_model.adapter_B[1]): p.data.copy_(ckpt["B1"][i])
print("  LoRA adapter weights loaded.")

for grp, key in [(0, "h0"), (1, "h1")]:
    saved_w  = ckpt[key]["weight"]
    saved_b  = ckpt[key]["bias"]
    pad_w    = torch.zeros(VOCAB_SIZE - saved_w.shape[0], saved_w.shape[1])
    pad_b    = torch.zeros(VOCAB_SIZE - saved_b.shape[0])
    padded_w = torch.cat([saved_w, pad_w], dim=0)
    padded_b = torch.cat([saved_b, pad_b], dim=0)
    lora_model.ctc_heads[grp].weight.data.copy_(padded_w)
    lora_model.ctc_heads[grp].bias.data.copy_(padded_b)
    print(f"  CTC head {grp}: padded {saved_w.shape[0]} → {VOCAB_SIZE} tokens")

lora_model = lora_model.to(device)
b2_model   = b2_model.to(device)
lora_model.eval()
lora_model._load_group(0)

print(f"\nModel ready on {device}.")
print(f"CTC head shape: {lora_model.ctc_heads[0].weight.shape}")
print(f"LoRA layers:    {len(lora_model._lora_layers)}")

Building model...
Injecting LoRA...
  Replaced 0 linear layers
  24 LoRALinear layers found

Loading saved weights from /kaggle/input/datasets/kavishk/2-group-lora-weights/p1v6_two_group_lora.pt...
  LoRA adapter weights loaded.
  CTC head 0: padded 30 → 32 tokens
  CTC head 1: padded 30 → 32 tokens

Model ready on cuda.
CTC head shape: torch.Size([32, 768])
LoRA layers:    24


## Step 6 — Load Severity MLP

In [16]:
class SeverityMLP(nn.Module):
    def __init__(self, hidden=768, n_cls=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),    nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, n_cls),
        )
    def forward(self, x): return self.net(x)


severity_mlp = SeverityMLP().to(device)
ckpt_mlp     = torch.load(MLP_PT, map_location="cpu")
severity_mlp.load_state_dict(
    ckpt_mlp["state_dict"] if "state_dict" in ckpt_mlp else ckpt_mlp
)
severity_mlp.eval()
if "best_val_acc" in ckpt_mlp:
    print(f"MLP loaded. Val acc at save: {ckpt_mlp['best_val_acc']*100:.2f}%")
else:
    print("MLP loaded.")

MLP loaded. Val acc at save: 95.32%


## Step 7 — Helper functions

In [17]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def decode_ctc_greedy(pred_ids, tokenizer):
    blank = tokenizer.pad_token_id
    out   = []
    for seq in pred_ids:
        col = [k for k, _ in groupby(seq)]
        fil = [t for t in col if t != blank]
        out.append(tokenizer.decode(fil, skip_special_tokens=True) if fil else "")
    return out


def get_layer6_pooled(model, iv, am):
    with torch.no_grad():
        out = model.wavlm(iv, attention_mask=am, output_hidden_states=True)
    l6 = out.hidden_states[7]
    if am is not None:
        fl   = model.wavlm._get_feat_extract_output_lengths(am.sum(-1)).long()
        T    = l6.size(1)
        mask = (torch.arange(T, device=device).unsqueeze(0)
                < fl.unsqueeze(1)).unsqueeze(-1).float()
        return (l6 * mask).sum(1) / mask.sum(1).clamp(min=1)
    return l6.mean(dim=1)


print("Helpers ready.")

Helpers ready.


## Step 8 — Collect logits (test + val)

In [26]:
def collect_with_b3_style(dataset_raw, cats_list, desc=""):
    """
    Collect logits using raw audio processing — identical to B3.
    dataset_raw: the raw torgo_test HuggingFace dataset (not preprocessed)
    """
    lora_model.eval()
    lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list = [], [], [], [], [], []

    for i in range(0, len(dataset_raw), BATCH):
        end     = min(i + BATCH, len(dataset_raw))
        samples = [dataset_raw[j] for j in range(i, end)]
        samples = [s for s in samples
                   if len(s["audio"]["array"]) <= MAX_AUDIO]
        if not samples: continue

        # Exact same processing as B3
        arrays = [s["audio"]["array"] for s in samples]
        sr     = samples[0]["audio"]["sampling_rate"]
        inputs = processor(arrays, sampling_rate=sr,
                           return_tensors="pt", padding=True)
        iv     = inputs.input_values.to(device)
        am     = inputs.attention_mask.to(device) \
                 if "attention_mask" in inputs else None

        with torch.no_grad():
            lora_model.set_active_group(0)
            out0 = lora_model.encoder(iv, attention_mask=am,
                                      output_hidden_states=True)
            lora_model.set_active_group(1)
            out1 = lora_model.encoder(iv, attention_mask=am,
                                      output_hidden_states=True)

            lg0 = lora_model.ctc_heads[0](out0.last_hidden_state)
            lg1 = lora_model.ctc_heads[1](out1.last_hidden_state)

            pooled = get_layer6_pooled(b2_model, iv, am)
            sp     = F.softmax(severity_mlp(pooled), dim=-1).cpu().numpy()

        for j, s in enumerate(samples):
            w0 = float(sp[j][0] + sp[j][1])
            w1 = float(sp[j][2] + sp[j][3])
            lg0_list.append(lg0[j].cpu().numpy())
            lg1_list.append(lg1[j].cpu().numpy())
            gw_list.append([w0, w1])
            lab_list.append(s["transcription"].lower().strip())
            cat_list.append(s["Category"])

            # Greedy from dominant adapter
            lg_dom = lg0[j].cpu() if w0 >= w1 else lg1[j].cpu()
            pid    = lg_dom.argmax(dim=-1).numpy()
            text   = decode_ctc_greedy([pid], processor.tokenizer)[0]
            greedy_list.append(text.strip())

        if (i // BATCH + 1) % 20 == 0:
            print(f"  [{desc}] {end}/{len(dataset_raw)}")

    return lg0_list, lg1_list, gw_list, lab_list, cat_list, greedy_list


print("Collecting test logits (B3-style raw audio)...")
(all_lg0, all_lg1, all_gw,
 all_labels, all_cats, greedy_preds) = collect_with_b3_style(
    torgo_test, test_cats_filtered, "test"
)

print("\nCollecting val logits (B3-style raw audio)...")
(val_lg0, val_lg1, val_gw,
 val_labels, val_cats, _) = collect_with_b3_style(
    torgo_val, val_cats_filtered, "val"
)

eg     = [p if p else "⟨empty⟩" for p in greedy_preds]
gw_wer = wer_metric.compute(predictions=eg, references=all_labels)
gw_cer = cer_metric.compute(predictions=eg, references=all_labels)
res_df = pd.DataFrame({"prediction": greedy_preds,
                        "reference":  all_labels, "Category": all_cats})

print(f"\nP1v6 Greedy (raw audio, dominant adapter): "
      f"WER={gw_wer*100:.2f}%  CER={gw_cer*100:.2f}%")
for cat in ["control", "mild", "moderate", "severe"]:
    sub = res_df[res_df["Category"] == cat]
    sub = sub[sub["reference"].str.strip().str.len() > 0]
    if not len(sub): continue
    pg  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
    gw  = wer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    gc  = cer_metric.compute(predictions=pg, references=sub["reference"].tolist())
    print(f"  {cat:10s}: WER={gw*100:.2f}%  CER={gc*100:.2f}%  (n={len(sub)})")

  [test] 160/1786
  [test] 320/1786
  [test] 480/1786
  [test] 640/1786
  [test] 800/1786
  [test] 960/1786
  [test] 1120/1786
  [test] 1280/1786
  [test] 1440/1786
  [test] 1600/1786
  [test] 1760/1786

  [val] 160/1111
  [val] 320/1111
  [val] 480/1111
  [val] 640/1111
  [val] 800/1111
  [val] 960/1111

P1v6 Greedy (raw audio, dominant adapter): WER=49.40%  CER=23.25%
  control   : WER=25.59%  CER=7.94%  (n=302)
  mild      : WER=31.18%  CER=9.56%  (n=673)
  moderate  : WER=76.35%  CER=41.90%  (n=575)
  severe    : WER=71.99%  CER=42.49%  (n=236)


## Step 9 — Build KenLM decoder (vocab size 32)

In [28]:
lm_dir = "/kaggle/temp/kenlm"
os.makedirs(lm_dir, exist_ok=True)
lm_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/lm.binary", cache_dir=lm_dir,
)
uni_path = hf_hub_download(
    repo_id="jonatasgrosman/wav2vec2-large-xlsr-53-english",
    filename="language_model/unigrams.txt", cache_dir=lm_dir,
)
unigrams = open(uni_path).read().strip().split("\n")
print(f"KenLM ready. Unigrams: {len(unigrams):,}")

# Build vocab_list at full size 32 — matches logit dimension
vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [None] * len(vocab_dict)  # 32
for tok, idx in vocab_dict.items():
    vocab_list[idx] = tok
vocab_list[processor.tokenizer.pad_token_id] = ""  # CTC blank

print(f"vocab_list size: {len(vocab_list)}")
print(f"vocab_list: {vocab_list}")

_dec_cache = {}

def get_decoder(alpha, beta):
    key = (round(float(alpha), 3), round(float(beta), 3))
    if key not in _dec_cache:
        _dec_cache[key] = build_ctcdecoder(
            labels=vocab_list,
            kenlm_model_path=lm_path,
            unigrams=unigrams,
            alpha=key[0],
            beta=key[1],
        )
    return _dec_cache[key]


def to_log_probs(lg):
    p = np.exp(lg) / np.exp(lg).sum(axis=-1, keepdims=True)
    return np.log(np.clip(p, 1e-8, 1.0))


def decode_hard(lg0, lg1, gw, alpha, beta, beam_width=100):
    """Hard routing — use dominant adapter's logits for beam search."""
    lg = lg0 if gw[0] >= gw[1] else lg1
    return get_decoder(alpha, beta).decode(
        to_log_probs(lg), beam_width=beam_width
    ).strip().lower()


def decode_best_beam(lg0, lg1, gw, alpha, beta, beam_width=100, n_best=10):
    """
    Run beam search on both adapters independently.
    Weight top hypothesis scores by MLP probs, pick best.
    No logit blending — sharp acoustic signal for each adapter.
    """
    dec = get_decoder(alpha, beta)

    beams0 = dec.decode_beams(to_log_probs(lg0), beam_width=beam_width)[:n_best]
    beams1 = dec.decode_beams(to_log_probs(lg1), beam_width=beam_width)[:n_best]

    text0  = beams0[0][0] if beams0 else ""
    text1  = beams1[0][0] if beams1 else ""
    score0 = (beams0[0][3] + beams0[0][4]) if beams0 else -1e9
    score1 = (beams1[0][3] + beams1[0][4]) if beams1 else -1e9

    return (text0 if gw[0] * score0 >= gw[1] * score1 else text1).strip().lower()


print("Decoder helpers ready.")

KenLM ready. Unigrams: 373,978
vocab_list size: 32
vocab_list: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', "'", '|', '[UNK]', '', '<s>', '</s>']
Decoder helpers ready.


In [29]:
def decode_oracle(lg0, lg1, true_cat, alpha, beta, beam_width=100):
    """Oracle routing — true severity label picks adapter. Upper bound."""
    lg = lg0 if SEV_TO_GROUP[true_cat] == 0 else lg1
    return get_decoder(alpha, beta).decode(
        to_log_probs(lg), beam_width=beam_width
    ).strip().lower()

In [30]:
best_beta = 1.989
best_alpha = 0.459
BEAM_WIDTH = 100

print(f"Final eval: alpha={best_alpha}  beta={best_beta}  beam={BEAM_WIDTH}")

strategies = [
    ("hard_routing",   lambda i: decode_hard(
        all_lg0[i], all_lg1[i], all_gw[i], best_alpha, best_beta, BEAM_WIDTH)),
    ("best_beam",      lambda i: decode_best_beam(
        all_lg0[i], all_lg1[i], all_gw[i], best_alpha, best_beta, BEAM_WIDTH)),
    ("oracle_routing", lambda i: decode_oracle(
        all_lg0[i], all_lg1[i], all_cats[i], best_alpha, best_beta, BEAM_WIDTH)),
]

for name, decode_fn in strategies:
    print(f"\nRunning {name}...")
    preds = [decode_fn(i) for i in range(len(all_lg0))]
    ep    = [p if p else "⟨empty⟩" for p in preds]
    wer   = wer_metric.compute(predictions=ep, references=all_labels)
    cer   = cer_metric.compute(predictions=ep, references=all_labels)
    df_   = pd.DataFrame({"prediction": preds,
                           "reference":  all_labels, "Category": all_cats})
    print(f"  WER={wer*100:.2f}%  CER={cer*100:.2f}%  "
          f"Improvement={( gw_wer-wer)*100:.2f}pp  vs B3={(wer-0.3236)*100:+.2f}pp")
    for cat in ["control", "mild", "moderate", "severe"]:
        sub = df_[df_["Category"] == cat]
        sub = sub[sub["reference"].str.strip().str.len() > 0]
        if not len(sub): continue
        pf  = [p if p else "⟨empty⟩" for p in sub["prediction"].tolist()]
        fw  = wer_metric.compute(predictions=pf, references=sub["reference"].tolist())
        fc  = cer_metric.compute(predictions=pf, references=sub["reference"].tolist())
        print(f"    {cat:10s}: WER={fw*100:.2f}%  CER={fc*100:.2f}%")
    if name == "best_beam":
        df_.to_csv("/kaggle/working/p1v6_kenlm_results.csv", index=False)

Final eval: alpha=0.459  beta=1.989  beam=100

Running hard_routing...
  WER=31.06%  CER=17.93%  Improvement=18.34pp  vs B3=-1.30pp
    control   : WER=11.18%  CER=4.51%
    mild      : WER=11.14%  CER=4.79%
    moderate  : WER=57.22%  CER=34.57%
    severe    : WER=55.85%  CER=37.82%

Running best_beam...
  WER=31.10%  CER=18.31%  Improvement=18.29pp  vs B3=-1.26pp
    control   : WER=10.74%  CER=4.45%
    mild      : WER=11.20%  CER=4.73%
    moderate  : WER=57.85%  CER=35.83%
    severe    : WER=55.14%  CER=38.08%

Running oracle_routing...
  WER=30.96%  CER=17.90%  Improvement=18.44pp  vs B3=-1.40pp
    control   : WER=11.03%  CER=4.51%
    mild      : WER=10.96%  CER=4.70%
    moderate  : WER=57.22%  CER=34.58%
    severe    : WER=55.85%  CER=37.82%
